### Model tuning


Tune the best tree-based models on the advanced men's and women's datasets.

In [49]:
import pandas as pd

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix, roc_auc_score, log_loss, brier_score_loss

from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

import json
import joblib
from pathlib import Path

In [7]:
men_df = pd.read_csv("../data/processed/m_tournament_training_dataset_advanced.csv")
women_df = pd.read_csv("../data/processed/w_tournament_training_dataset_advanced.csv")

In [8]:
def evaluate_binary_classifier(model, X_test, y_test, threshold=0.5, print_results=True):
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        scores = model.decision_function(X_test)
        # convert scores to 0-1 range approximately
        y_prob = (scores - scores.min()) / (scores.max() - scores.min())
    else:
        raise ValueError("Model does not support predict_proba or decision_function.")

    y_pred = (y_prob >= threshold).astype(int)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)

    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred)

    logloss = log_loss(y_test, y_prob)
    brier = brier_score_loss(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)

    results = {
        "accuracy": acc,
        "f1_score": f1,
        "precision": precision,
        "recall": recall,
        "classification_report": report,
        "confusion_matrix": cm,
        "log_loss": logloss,
        "brier_score": brier,
        "auc": auc
    }

    if print_results:
        print("=== Classification Metrics ===")
        print(f"Accuracy:   {acc:.4f}")
        print(f"F1 Score:   {f1:.4f}")
        print(f"Precision:  {precision:.4f}")
        print(f"Recall:     {recall:.4f}")

        print("\n=== Classification Report ===")
        print(report)

        print("=== Confusion Matrix ===")
        print(cm)

        print("\n=== Probability Metrics ===")
        print(f"Log Loss:   {logloss:.4f}")
        print(f"Brier Score:{brier:.4f}")
        print(f"AUC:        {auc:.4f}")

    return results

### Helper function to store results

In [9]:
def add_model_result(model_results, dataset_name, model_name, cv_logloss, results):
    model_results.append({
        "Dataset": dataset_name,
        "Model": model_name,
        "CV_LogLoss": cv_logloss,
        "Accuracy": results["accuracy"],
        "F1": results["f1_score"],
        "Precision": results["precision"],
        "Recall": results["recall"],
        "LogLoss": results["log_loss"],
        "BrierScore": results["brier_score"],
        "AUC": results["auc"]
    })

### Selected features

In [10]:
selected_cols = [
    "Season",
     "Target",
    "SeedNumDiff",
    "RankingDiff",
    "MarginDiff",
    "NetRatingDiff",
    "OffEffDiff",
    "DefEffDiff",
    "WinPctDiff",
    "OffDefGap",
    "DominanceScore",
    "NetRating_Margin_Interaction",
    "Margin_Ranking_Interaction",
    "TurnoverMarginDiff",
    "ReboundPctDiff"
]

### Split men data

In [11]:
men_df = men_df[selected_cols].copy()

In [12]:
season_cutoff = 2023
drop_cols = ["Season", "Target"]

In [13]:
men_train_df = men_df[men_df["Season"] < season_cutoff].copy()
men_test_df = men_df[men_df["Season"] >= season_cutoff].copy()

In [14]:
X_train_men = men_train_df.drop(columns=drop_cols)
y_train_men = men_train_df["Target"]

X_test_men = men_test_df.drop(columns=drop_cols)
y_test_men = men_test_df["Target"]

### Split women data

In [15]:
women_df = women_df[selected_cols].copy()

In [16]:
women_train_df = women_df[women_df["Season"] < season_cutoff].copy()
women_test_df = women_df[women_df["Season"] >= season_cutoff].copy()

In [17]:
X_train_women = women_train_df.drop(columns=drop_cols)
y_train_women = women_train_df["Target"]

X_test_women = women_test_df.drop(columns=drop_cols)
y_test_women = women_test_df["Target"]

### Cross validation strategy

In [18]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

### Tuning functions

In [19]:
def tune_random_forest(X_train, y_train, X_test, y_test):
    rf_model = RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    )

    rf_param_grid = {
        "n_estimators": [200, 300, 500],
        "max_depth": [4, 5, 6, 8],
        "min_samples_split": [5, 10, 15],
        "min_samples_leaf": [2, 3, 5]
    }

    rf_grid = GridSearchCV(
        estimator=rf_model,
        param_grid=rf_param_grid,
        scoring="neg_log_loss",
        cv=cv,
        n_jobs=-1,
        verbose=2,
        refit=True
    )

    rf_grid.fit(X_train, y_train)

    print("Best RF Params:")
    print(rf_grid.best_params_)
    print("Best RF CV Score (neg log loss):")
    print(rf_grid.best_score_)
    print("Best RF CV Log Loss:")
    print(-rf_grid.best_score_)

    best_rf = rf_grid.best_estimator_
    rf_results = evaluate_binary_classifier(best_rf, X_test, y_test)

    return best_rf, rf_grid, rf_results

In [20]:
def tune_xgboost(X_train, y_train, X_test, y_test):
    xgb_model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42
    )

    xgb_param_grid = {
        "n_estimators": [200, 400, 600],
        "max_depth": [2, 3, 4],
        "learning_rate": [0.03, 0.05, 0.07],
        "subsample": [0.8, 0.9, 1.0],
        "colsample_bytree": [0.8, 0.9, 1.0],
        "reg_lambda": [1.0, 1.5, 2.0]
    }

    xgb_grid = GridSearchCV(
        estimator=xgb_model,
        param_grid=xgb_param_grid,
        scoring="neg_log_loss",
        cv=cv,
        n_jobs=-1,
        verbose=2,
        refit=True
    )

    xgb_grid.fit(X_train, y_train)

    print("Best XGB Params:")
    print(xgb_grid.best_params_)
    print("Best XGB CV Score (neg log loss):")
    print(xgb_grid.best_score_)
    print("Best XGB CV Log Loss:")
    print(-xgb_grid.best_score_)

    best_xgb = xgb_grid.best_estimator_
    xgb_results = evaluate_binary_classifier(best_xgb, X_test, y_test)

    return best_xgb, xgb_grid, xgb_results

In [21]:
def tune_voting_classifier(best_rf, best_xgb, X_train, y_train, X_test, y_test):
    voting_model = VotingClassifier(
        estimators=[
            ("rf", best_rf),
            ("xgb", best_xgb)
        ],
        voting="soft"
    )

    voting_model.fit(X_train, y_train)
    voting_results = evaluate_binary_classifier(voting_model, X_test, y_test)

    return voting_model, voting_results

### Men tuning

In [22]:
men_model_results = []

In [23]:
print("\nMen - Random Forest Tuning")
men_best_rf, men_rf_grid, men_rf_results = tune_random_forest(
    X_train_men, y_train_men, X_test_men, y_test_men
)
add_model_result(
    men_model_results,
    "Men",
    "Random Forest Tuned",
    -men_rf_grid.best_score_,
    men_rf_results
)


Men - Random Forest Tuning
Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best RF Params:
{'max_depth': 8, 'min_samples_leaf': 5, 'min_samples_split': 15, 'n_estimators': 300}
Best RF CV Score (neg log loss):
-0.5597439340320418
Best RF CV Log Loss:
0.5597439340320418
=== Classification Metrics ===
Accuracy:   0.7239
F1 Score:   0.7232
Precision:  0.7250
Recall:     0.7214

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.72      0.73      0.72       201
           1       0.72      0.72      0.72       201

    accuracy                           0.72       402
   macro avg       0.72      0.72      0.72       402
weighted avg       0.72      0.72      0.72       402

=== Confusion Matrix ===
[[146  55]
 [ 56 145]]

=== Probability Metrics ===
Log Loss:   0.5424
Brier Score:0.1841
AUC:        0.7948


In [24]:
print("\nMen - XGBoost Tuning")
men_best_xgb, men_xgb_grid, men_xgb_results = tune_xgboost(
    X_train_men, y_train_men, X_test_men, y_test_men
)
add_model_result(
    men_model_results,
    "Men",
    "XGBoost Tuned",
    -men_xgb_grid.best_score_,
    men_xgb_results
)


Men - XGBoost Tuning
Fitting 5 folds for each of 729 candidates, totalling 3645 fits
Best XGB Params:
{'colsample_bytree': 1.0, 'learning_rate': 0.03, 'max_depth': 2, 'n_estimators': 200, 'reg_lambda': 2.0, 'subsample': 0.9}
Best XGB CV Score (neg log loss):
-0.55653981530174
Best XGB CV Log Loss:
0.55653981530174
=== Classification Metrics ===
Accuracy:   0.7289
F1 Score:   0.7295
Precision:  0.7277
Recall:     0.7313

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.73      0.73      0.73       201
           1       0.73      0.73      0.73       201

    accuracy                           0.73       402
   macro avg       0.73      0.73      0.73       402
weighted avg       0.73      0.73      0.73       402

=== Confusion Matrix ===
[[146  55]
 [ 54 147]]

=== Probability Metrics ===
Log Loss:   0.5472
Brier Score:0.1850
AUC:        0.7944


In [25]:
print("\nMen - Voting Classifier")
men_voting_model, men_voting_results = tune_voting_classifier(
    men_best_rf, men_best_xgb,
    X_train_men, y_train_men,
    X_test_men, y_test_men
)
add_model_result(
    men_model_results,
    "Men",
    "Voting Classifier Tuned",
    None,
    men_voting_results
)


Men - Voting Classifier
=== Classification Metrics ===
Accuracy:   0.7289
F1 Score:   0.7282
Precision:  0.7300
Recall:     0.7264

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.73      0.73      0.73       201
           1       0.73      0.73      0.73       201

    accuracy                           0.73       402
   macro avg       0.73      0.73      0.73       402
weighted avg       0.73      0.73      0.73       402

=== Confusion Matrix ===
[[147  54]
 [ 55 146]]

=== Probability Metrics ===
Log Loss:   0.5432
Brier Score:0.1840
AUC:        0.7972


In [26]:
men_results_df = pd.DataFrame(men_model_results).sort_values(
    by=["LogLoss", "BrierScore", "AUC"],
    ascending=[True, True, False]
).reset_index(drop=True)

### Women tuning

In [27]:
women_model_results = []

In [28]:
print("\nWomen - Random Forest Tuning")
women_best_rf, women_rf_grid, women_rf_results = tune_random_forest(
    X_train_women, y_train_women, X_test_women, y_test_women
)
add_model_result(
    women_model_results,
    "Women",
    "Random Forest Tuned",
    -women_rf_grid.best_score_,
    women_rf_results
)


Women - Random Forest Tuning
Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best RF Params:
{'max_depth': 5, 'min_samples_leaf': 5, 'min_samples_split': 5, 'n_estimators': 300}
Best RF CV Score (neg log loss):
-0.426770417769616
Best RF CV Log Loss:
0.426770417769616
=== Classification Metrics ===
Accuracy:   0.7910
F1 Score:   0.7900
Precision:  0.7940
Recall:     0.7861

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.79      0.80      0.79       201
           1       0.79      0.79      0.79       201

    accuracy                           0.79       402
   macro avg       0.79      0.79      0.79       402
weighted avg       0.79      0.79      0.79       402

=== Confusion Matrix ===
[[160  41]
 [ 43 158]]

=== Probability Metrics ===
Log Loss:   0.4231
Brier Score:0.1390
AUC:        0.8857


In [29]:
print("\nWomen - XGBoost Tuning")
women_best_xgb, women_xgb_grid, women_xgb_results = tune_xgboost(
    X_train_women, y_train_women, X_test_women, y_test_women
)
add_model_result(
    women_model_results,
    "Women",
    "XGBoost Tuned",
    -women_xgb_grid.best_score_,
    women_xgb_results
)


Women - XGBoost Tuning
Fitting 5 folds for each of 729 candidates, totalling 3645 fits
Best XGB Params:
{'colsample_bytree': 1.0, 'learning_rate': 0.03, 'max_depth': 2, 'n_estimators': 200, 'reg_lambda': 1.0, 'subsample': 1.0}
Best XGB CV Score (neg log loss):
-0.4094463536453567
Best XGB CV Log Loss:
0.4094463536453567
=== Classification Metrics ===
Accuracy:   0.7910
F1 Score:   0.7910
Precision:  0.7910
Recall:     0.7910

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.79      0.79      0.79       201
           1       0.79      0.79      0.79       201

    accuracy                           0.79       402
   macro avg       0.79      0.79      0.79       402
weighted avg       0.79      0.79      0.79       402

=== Confusion Matrix ===
[[159  42]
 [ 42 159]]

=== Probability Metrics ===
Log Loss:   0.4122
Brier Score:0.1396
AUC:        0.8786


In [30]:
print("\nWomen - Voting Classifier")
women_voting_model, women_voting_results = tune_voting_classifier(
    women_best_rf, women_best_xgb,
    X_train_women, y_train_women,
    X_test_women, y_test_women
)
add_model_result(
    women_model_results,
    "Women",
    "Voting Classifier Tuned",
    None,
    women_voting_results
)


Women - Voting Classifier
=== Classification Metrics ===
Accuracy:   0.7910
F1 Score:   0.7910
Precision:  0.7910
Recall:     0.7910

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.79      0.79      0.79       201
           1       0.79      0.79      0.79       201

    accuracy                           0.79       402
   macro avg       0.79      0.79      0.79       402
weighted avg       0.79      0.79      0.79       402

=== Confusion Matrix ===
[[159  42]
 [ 42 159]]

=== Probability Metrics ===
Log Loss:   0.4162
Brier Score:0.1385
AUC:        0.8849


In [31]:
women_results_df = pd.DataFrame(women_model_results).sort_values(
    by=["LogLoss", "BrierScore", "AUC"],
    ascending=[True, True, False]
).reset_index(drop=True)

### Comparison tables

In [32]:
men_comparison_df = pd.DataFrame([
    {
        "Dataset": "Men",
        "Model": "Random Forest Tuned",
        "CV LogLoss": -men_rf_grid.best_score_,
        "Test Accuracy": men_rf_results["accuracy"],
        "Test F1": men_rf_results["f1_score"],
        "Test Precision": men_rf_results["precision"],
        "Test Recall": men_rf_results["recall"],
        "Test LogLoss": men_rf_results["log_loss"],
        "Test Brier": men_rf_results["brier_score"],
        "Test AUC": men_rf_results["auc"]
    },
    {
        "Dataset": "Men",
        "Model": "XGBoost Tuned",
        "CV LogLoss": -men_xgb_grid.best_score_,
        "Test Accuracy": men_xgb_results["accuracy"],
        "Test F1": men_xgb_results["f1_score"],
        "Test Precision": men_xgb_results["precision"],
        "Test Recall": men_xgb_results["recall"],
        "Test LogLoss": men_xgb_results["log_loss"],
        "Test Brier": men_xgb_results["brier_score"],
        "Test AUC": men_xgb_results["auc"]
    }
]).sort_values(by="Test LogLoss").reset_index(drop=True)

In [33]:
women_comparison_df = pd.DataFrame([
    {
        "Dataset": "Women",
        "Model": "Random Forest Tuned",
        "CV LogLoss": -women_rf_grid.best_score_,
        "Test Accuracy": women_rf_results["accuracy"],
        "Test F1": women_rf_results["f1_score"],
        "Test Precision": women_rf_results["precision"],
        "Test Recall": women_rf_results["recall"],
        "Test LogLoss": women_rf_results["log_loss"],
        "Test Brier": women_rf_results["brier_score"],
        "Test AUC": women_rf_results["auc"]
    },
    {
        "Dataset": "Women",
        "Model": "XGBoost Tuned",
        "CV LogLoss": -women_xgb_grid.best_score_,
        "Test Accuracy": women_xgb_results["accuracy"],
        "Test F1": women_xgb_results["f1_score"],
        "Test Precision": women_xgb_results["precision"],
        "Test Recall": women_xgb_results["recall"],
        "Test LogLoss": women_xgb_results["log_loss"],
        "Test Brier": women_xgb_results["brier_score"],
        "Test AUC": women_xgb_results["auc"]
    }
]).sort_values(by="Test LogLoss").reset_index(drop=True)

In [34]:
print(men_comparison_df)

  Dataset                Model  CV LogLoss  Test Accuracy   Test F1  \
0     Men  Random Forest Tuned    0.559744       0.723881  0.723192   
1     Men        XGBoost Tuned    0.556540       0.728856  0.729529   

   Test Precision  Test Recall  Test LogLoss  Test Brier  Test AUC  
0        0.725000     0.721393      0.542386    0.184149  0.794832  
1        0.727723     0.731343      0.547165    0.184989  0.794362  


In [35]:
print(women_comparison_df)

  Dataset                Model  CV LogLoss  Test Accuracy   Test F1  \
0   Women        XGBoost Tuned    0.409446       0.791045  0.791045   
1   Women  Random Forest Tuned    0.426770       0.791045  0.790000   

   Test Precision  Test Recall  Test LogLoss  Test Brier  Test AUC  
0        0.791045     0.791045      0.412230    0.139573  0.878555  
1        0.793970     0.786070      0.423092    0.139045  0.885745  


### Feature importance

In [36]:
men_rf_importance_df = pd.DataFrame({
    "Feature": X_train_men.columns,
    "Importance": men_best_rf.feature_importances_
}).sort_values(by="Importance", ascending=False).reset_index(drop=True)

In [37]:
men_xgb_importance_df = pd.DataFrame({
    "Feature": X_train_men.columns,
    "Importance": men_best_xgb.feature_importances_
}).sort_values(by="Importance", ascending=False).reset_index(drop=True)

In [38]:
women_rf_importance_df = pd.DataFrame({
    "Feature": X_train_women.columns,
    "Importance": women_best_rf.feature_importances_
}).sort_values(by="Importance", ascending=False).reset_index(drop=True)


In [39]:
women_xgb_importance_df = pd.DataFrame({
    "Feature": X_train_women.columns,
    "Importance": women_best_xgb.feature_importances_
}).sort_values(by="Importance", ascending=False).reset_index(drop=True)

In [40]:
print("\nMen - Random Forest Feature Importances")
print(men_rf_importance_df.head(20))


Men - Random Forest Feature Importances
                         Feature  Importance
0                    SeedNumDiff    0.225757
1                    RankingDiff    0.130201
2                      OffDefGap    0.101822
3                     MarginDiff    0.097487
4                  NetRatingDiff    0.090861
5     Margin_Ranking_Interaction    0.052562
6                     WinPctDiff    0.048724
7   NetRating_Margin_Interaction    0.046620
8                     OffEffDiff    0.044830
9             TurnoverMarginDiff    0.042297
10                DominanceScore    0.041190
11                    DefEffDiff    0.040497
12                ReboundPctDiff    0.037149


In [41]:
print("\nMen - XGBoost Feature Importances")
print(men_xgb_importance_df.head(20))


Men - XGBoost Feature Importances
                         Feature  Importance
0                    SeedNumDiff    0.369045
1                  NetRatingDiff    0.145422
2                     MarginDiff    0.077599
3                    RankingDiff    0.060055
4   NetRating_Margin_Interaction    0.057112
5                 DominanceScore    0.048229
6                 ReboundPctDiff    0.045199
7     Margin_Ranking_Interaction    0.044515
8             TurnoverMarginDiff    0.044076
9                     DefEffDiff    0.039087
10                    OffEffDiff    0.039008
11                    WinPctDiff    0.030653
12                     OffDefGap    0.000000


In [42]:
print("\nWomen - Random Forest Feature Importances")
print(women_rf_importance_df.head(20))


Women - Random Forest Feature Importances
                         Feature  Importance
0                    RankingDiff    0.346209
1                    SeedNumDiff    0.237016
2                     MarginDiff    0.090015
3     Margin_Ranking_Interaction    0.081261
4                      OffDefGap    0.059719
5                  NetRatingDiff    0.055299
6                     OffEffDiff    0.030912
7                 ReboundPctDiff    0.024450
8                     WinPctDiff    0.020624
9                 DominanceScore    0.016054
10  NetRating_Margin_Interaction    0.015507
11                    DefEffDiff    0.013966
12            TurnoverMarginDiff    0.008969


In [43]:
print("\nWomen - XGBoost Feature Importances")
print(women_xgb_importance_df.head(20))


Women - XGBoost Feature Importances
                         Feature  Importance
0                    RankingDiff    0.571833
1                    SeedNumDiff    0.094219
2                  NetRatingDiff    0.052338
3             TurnoverMarginDiff    0.049018
4                     MarginDiff    0.047519
5                     WinPctDiff    0.046369
6                 ReboundPctDiff    0.039486
7     Margin_Ranking_Interaction    0.036836
8                     DefEffDiff    0.033044
9   NetRating_Margin_Interaction    0.029338
10                    OffEffDiff    0.000000
11                DominanceScore    0.000000
12                     OffDefGap    0.000000


### Save outputs


In [44]:
men_results_df.to_csv("../data/processed/model_comparison_results_tuned_men.csv", index=False)
women_results_df.to_csv("../data/processed/model_comparison_results_tuned_women.csv", index=False)

In [45]:
men_comparison_df.to_csv("../data/processed/tuned_model_comparison_men.csv", index=False)
women_comparison_df.to_csv("../data/processed/tuned_model_comparison_women.csv", index=False)

In [46]:
men_xgb_importance_df.to_csv("../data/processed/tuned_xgb_feature_importance_men.csv", index=False)
men_rf_importance_df.to_csv("../data/processed/tuned_rf_feature_importance_men.csv", index=False)

In [47]:
women_xgb_importance_df.to_csv("../data/processed/tuned_xgb_feature_importance_women.csv", index=False)
women_rf_importance_df.to_csv("../data/processed/tuned_rf_feature_importance_women.csv", index=False)

### Saving the model

In [50]:
artifacts_dir = Path("../artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)

- For men

In [51]:
men_model_lookup = {
    "Random Forest Tuned": men_best_rf,
    "XGBoost Tuned": men_best_xgb,
    "Voting Classifier Tuned": men_voting_model
}

men_best_model_name = men_results_df.iloc[0]["Model"]
men_best_model = men_model_lookup[men_best_model_name]

print("Best men model:", men_best_model_name)

Best men model: Random Forest Tuned


In [52]:
men_feature_cols = X_train_men.columns.tolist()

joblib.dump(men_best_model, artifacts_dir / "best_model_men.pkl")

with open(artifacts_dir / "best_model_men_features.json", "w") as f:
    json.dump(men_feature_cols, f)

- For women

In [53]:
women_model_lookup = {
    "Random Forest Tuned": women_best_rf,
    "XGBoost Tuned": women_best_xgb,
    "Voting Classifier Tuned": women_voting_model
}

women_best_model_name = women_results_df.iloc[0]["Model"]
women_best_model = women_model_lookup[women_best_model_name]

print("Best women model:", women_best_model_name)

Best women model: XGBoost Tuned


In [54]:
women_feature_cols = X_train_women.columns.tolist()

joblib.dump(women_best_model, artifacts_dir / "best_model_women.pkl")

with open(artifacts_dir / "best_model_women_features.json", "w") as f:
    json.dump(women_feature_cols, f)